# Vocabulario y Tokenizer

En este notebook aplicaremos el algoritmo Byte Pair Encodign para crear nuestro propio vocabulario de tokens.

Aplicaremos BPE sobre el conjunto de entrenamiento

In [1]:
#Descargamos el dataset
import pandas as pd
from datasets import load_dataset

def filter_dataset(dataframe):
    """
    Filtra el dataset, mismo que en dataset.ipynb
    """
    df = dataframe.dropna().copy()
    df = df.drop_duplicates(subset=['en','es'])


    #Texto sin espacios y en minuscula
    en_text = df['en'].str.lower().str.strip() 
    es_text = df['es'].str.lower().str.strip()

    #Longitudes de las frases
    long_en = df['en'].str.split().str.len()
    long_es = df['es'].str.split().str.len()
    ratio = long_es / long_en

    #Mascaras
    not_empy = (en_text != "") & (es_text != "")
    not_equal = (en_text != es_text)
    ming_lon = (long_en >= 3) & (long_es >= 3)
    m_ratio = (ratio > 0.5) & (ratio < 2.0)


    #Filtro
    mask = not_empy & not_equal & ming_lon & m_ratio
    return df[mask]


dataset = load_dataset("opus100", "en-es")
train = pd.DataFrame(dataset['train']['translation'])
train = filter_dataset(train)

Una vez cargado el dataset aplicaremos el algoritmo.

Crearemos un vocabulario de 16.000 tokens

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from config import VOCAB_PATH
import os

#Instancia
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False) #ByteLevel para no perder los espacios
tokenizer.decoder = ByteLevelDecoder()


#Objeto de la clase BPE
entrenador = BpeTrainer(
    vocab_size=16000, 
    special_tokens=["<PAD>", "<START>", "<END>", "<UNK>"],
    initial_alphabet=ByteLevel.alphabet() #añadimos alfabeto inicial al vocabulario
)


def iterator(data):
    for translation in data:
        yield translation['en']
        yield translation['es']

#Aplicamos BPE
tokenizer.train_from_iterator(iterator(train), trainer=entrenador)
tokenizer.save(os.path.join(VOCAB_PATH,"vocab_16k.json"))